# 13 Random Numbers

| | |
|---|---|
| **Path** | NumPy |
| **Prerequisite** | 12 |

NumPy's random tools live on a Generator. You build one with `np.random.default_rng(seed)`. The same seed rebuilds the same sequence, which is what makes a lesson repeatable.

A Generator is not the older bundle of functions on `np.random` itself. New code uses the Generator. The older functions share one hidden global state, so a call in one notebook cell can change a later call that you thought was independent.


## Example 1 — A seeded roll stays inside its range

### Predict

`integers(1, 7, size=5)` draws five integers from 1 inclusive to 7 exclusive, so each roll is $1,2,3,4,5,$ or $6$. The seed fixes which five appear. The check that belongs in the prediction is the range and the shape, not a memorized list.

### Run


In [1]:
import numpy as np

# The seed makes this cell print the same rolls every time it runs.
rng = np.random.default_rng(1)
rolls = rng.integers(1, 7, size=5)
print(rolls)
print(rolls.shape)
print(rolls.min() >= 1 and rolls.max() <= 6)


[3 4 5 6 1]
(5,)
True


## Example 2 — A normal sample agrees with its parameters

### Predict

One hundred thousand draws from a normal distribution with mean $10$ and standard deviation $2$ will not hit those numbers exactly. Their sample mean should land within $0.05$ of $10$, and their sample standard deviation within $0.05$ of $2$. That is the law of large numbers, checked rather than derived.

$$
X \sim \mathcal{N}(10,\ 2^2).
$$

### Run


In [2]:
import numpy as np

rng = np.random.default_rng(0)
sample = rng.normal(loc=10, scale=2, size=100_000)
# ddof=0 is enough at this size; the tolerance is the claim.
print(abs(sample.mean() - 10) < 0.05)
print(abs(sample.std() - 2) < 0.05)


True
True


## Example 3 — Choose, shuffle a copy, and leave the original

### Predict

`choice` returns three of the four names, with replacement by default, so a repeat is allowed. `permutation` returns a reordering and does not edit the array you passed.

### Run


In [3]:
import numpy as np

rng = np.random.default_rng(2)
names = np.array(["Ada", "Grace", "Alan", "Linus"])
picked = rng.choice(names, size=3)
order = rng.permutation(names)
print(picked.shape)
print(set(picked.tolist()).issubset(set(names.tolist())))
print(sorted(order.tolist()) == sorted(names.tolist()))
# permutation built a new order. names is untouched.
print(names.tolist())


(3,)
True
True
['Ada', 'Grace', 'Alan', 'Linus']


## Example 4 — Two generators do not share a stream

### Predict

Two generators built with the same seed produce the same first draw. Advancing one of them does not advance the other. Their second draws therefore differ if you draw once more from only one of them. Here both are asked for a first integer, and those first integers agree.

### Run


In [4]:
# Example 4 — Two generators do not share a stream.
import numpy as np

left = np.random.default_rng(7)
right = np.random.default_rng(7)
print(left.integers(0, 100))
print(right.integers(0, 100))


94
94


## A pitfall

`np.random.seed` and `np.random.integers` use a process-wide stream. A Generator you built does not read that stream. Mixing the two styles in one program makes the sequence depend on call order in a way the seed on the Generator does not control. This cell uses only a Generator.

### Run


In [5]:
import numpy as np

rng = np.random.default_rng(7)
first = rng.integers(0, 100)
# A fresh generator with the same seed repeats the first draw.
again = np.random.default_rng(7).integers(0, 100)
print(first == again)


True


## Summary

Build a Generator with `default_rng(seed)` and draw from that object. `integers` excludes the high end, the same rule as `range`. A large normal sample comes back near the mean and scale you asked for. `permutation` returns a new order. Keep one style: the Generator, not the global `np.random` functions.
